<a href="https://colab.research.google.com/github/LarsVoermans/master-thesis-pead/blob/main/Boosting_Bagging.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#All imports
from sklearn.preprocessing import OrdinalEncoder
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier



In [ ]:
from google.colab import files
uploaded = files.upload()  # er verschijnt een knop om bestanden te kiezen

Saving train_feature_engineered.parquet to train_feature_engineered.parquet
Saving val_feature_engineered.parquet to val_feature_engineered.parquet


In [ ]:
#Loading the dataset
train = pd.read_parquet("train_feature_engineered.parquet")
val = pd.read_parquet("val_feature_engineered.parquet")

In [ ]:
#making 3 class return
# Train
conditions_train = [
    (train['Return'] > 3),
    (train['Return'] >= -3) & (train['Return'] <= 3),
    (train['Return'] < -3)
]

choices = [0, 1, 2]

train['Return_class_3'] = np.select(conditions_train, choices, default=np.nan)


# Validation
conditions_val = [
    (val['Return'] > 3),
    (val['Return'] >= -3) & (val['Return'] <= 3),
    (val['Return'] < -3)
]

val['Return_class_3'] = np.select(conditions_val, choices, default=np.nan)



In [ ]:
#making a return class
target = "Return"

X_train = train.drop(columns=["Return","Return_class","EPS_Estimate","EPS_Actual","Close_Before","Open_After","Return_class_3"])
y_train = train["Return_class_3"]

X_val = val.drop(columns=["Return","Return_class","EPS_Estimate","EPS_Actual","Close_Before","Open_After","Return_class_3"])
y_val = val["Return_class_3"]


In [ ]:
#Dropping colums
drop_cols = [
    'CUSIP',
    'Global Company Key',
    'Historical CRSP PERMNO Link to COMPUSTAT Record',
    'Ticker',
    'Date',
    'EarningsDate',
    'Fiscal year end',
    'Fiscal quarter end',
    'Year',
    'Month'
]

X_train = X_train.drop(columns=[c for c in drop_cols if c in X_train.columns])
X_val = X_val.drop(columns=[c for c in drop_cols if c in X_val.columns])

In [ ]:
#XGB Classifier
# Train+ val
X_all = pd.concat([X_train, X_val], axis=0)
y_all = pd.concat([y_train, y_val], axis=0)

X_all.reset_index(drop=True, inplace=True)
y_all.reset_index(drop=True, inplace=True)

# Stratified K-Fold
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

accuracy_list = []
macro_f1_list = []
fold_reports = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_all, y_all), 1):
    print(f"\n--- Fold {fold} ---")

    X_tr, X_val_fold = X_all.iloc[train_idx], X_all.iloc[val_idx]
    y_tr, y_val_fold = y_all.iloc[train_idx], y_all.iloc[val_idx]

    # XGB Classifier
    xgb_model = XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        enable_categorical=True,
        random_state=42
    )

    # Fit
    xgb_model.fit(X_tr, y_tr)

    # Predict
    y_pred = xgb_model.predict(X_val_fold)

    # Scores
    acc = accuracy_score(y_val_fold, y_pred)
    f1 = f1_score(y_val_fold, y_pred, average='macro')
    report = classification_report(y_val_fold, y_pred, output_dict=True)

    accuracy_list.append(acc)
    macro_f1_list.append(f1)
    fold_reports.append(report)

    print(f"Accuracy: {acc:.4f}")
    print(f"Macro F1: {f1:.4f}")
    print("Classification Report:")
    print(classification_report(y_val_fold, y_pred))

# Average scores
print("\n=== Average over all folds ===")
print(f"Average Accuracy: {np.mean(accuracy_list):.4f}")
print(f"Average Macro F1: {np.mean(macro_f1_list):.4f}")

# Average classification report per class
avg_report = {}
classes = y_all.unique()
for c in classes:
    avg_report[c] = {}
    metrics = ['precision', 'recall', 'f1-score', 'support']
    for m in metrics:
        avg_report[c][m] = np.mean([fold_reports[i][str(c)][m] for i in range(n_splits)])
avg_report['macro avg'] = {m: np.mean([fold_reports[i]['macro avg'][m] for i in range(n_splits)]) for m in metrics}
avg_report['weighted avg'] = {m: np.mean([fold_reports[i]['weighted avg'][m] for i in range(n_splits)]) for m in metrics}

print("\n=== Average Classification Report per class ===")
print(pd.DataFrame(avg_report).T)


--- Fold 1 ---
Accuracy: 0.6962
Macro F1: 0.4922
Classification Report:
              precision    recall  f1-score   support

         0.0       0.51      0.31      0.39       411
         1.0       0.74      0.93      0.82      1474
         2.0       0.49      0.18      0.27       370

    accuracy                           0.70      2255
   macro avg       0.58      0.48      0.49      2255
weighted avg       0.65      0.70      0.65      2255


--- Fold 2 ---
Accuracy: 0.6943
Macro F1: 0.4912
Classification Report:
              precision    recall  f1-score   support

         0.0       0.53      0.31      0.39       412
         1.0       0.74      0.93      0.82      1473
         2.0       0.45      0.19      0.26       369

    accuracy                           0.69      2254
   macro avg       0.57      0.47      0.49      2254
weighted avg       0.65      0.69      0.65      2254


--- Fold 3 ---
Accuracy: 0.6952
Macro F1: 0.4961
Classification Report:
              preci

In [ ]:
# -----------------------------
# LGBM classifier
# -----------------------------
X_all = pd.concat([X_train, X_val], axis=0)
y_all = pd.concat([y_train, y_val], axis=0)

X_all = X_all.copy()


X_all.columns = X_all.columns.str.replace(r'[^A-Za-z0-9_]+', '_', regex=True)

# -----------------------------
# Stratified K-Fold
# -----------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

accuracy_list = []
macro_f1_list = []
reports = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_all, y_all), 1):

    print(f"\n--- Fold {fold} ---")

    X_tr = X_all.iloc[train_idx]
    X_val_fold = X_all.iloc[val_idx]

    y_tr = y_all.iloc[train_idx]
    y_val_fold = y_all.iloc[val_idx]

    # LightGBM model
    lgbm_model = LGBMClassifier(
        objective='multiclass',
        num_class=3,
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    # Train
    lgbm_model.fit(X_tr, y_tr)

    # Predict
    y_pred = lgbm_model.predict(X_val_fold)

    # Metrics
    acc = accuracy_score(y_val_fold, y_pred)
    macro_f1 = f1_score(y_val_fold, y_pred, average='macro')

    print("Accuracy:", acc)
    print("Macro F1:", macro_f1)
    print(classification_report(y_val_fold, y_pred))

    accuracy_list.append(acc)
    macro_f1_list.append(macro_f1)

    reports.append(classification_report(y_val_fold, y_pred, output_dict=True))

# -----------------------------
# Average Result
# -----------------------------
print("\n===== AVERAGE RESULTS =====")
print("Average Accuracy:", np.mean(accuracy_list))
print("Average Macro F1:", np.mean(macro_f1_list))

# -----------------------------
# Average classification report
# -----------------------------
avg_report = {}

for label in reports[0].keys():

    if label == "accuracy":
        avg_report[label] = np.mean([r[label] for r in reports])
        continue

    avg_report[label] = {}

    for metric in reports[0][label].keys():
        avg_report[label][metric] = np.mean([r[label][metric] for r in reports])

print("\nAverage Classification Report per class:")
print(pd.DataFrame(avg_report).T)


--- Fold 1 ---
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.025249 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 28731
[LightGBM] [Info] Number of data points in the train set: 9016, number of used features: 126
[LightGBM] [Info] Start training from score -1.700653
[LightGBM] [Info] Start training from score -0.425405
[LightGBM] [Info] Start training from score -1.808311
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightG

In [ ]:
# -----------------------------
# Catboost model
# -----------------------------
X_all = pd.concat([X_train, X_val], axis=0)
y_all = pd.concat([y_train, y_val], axis=0)

X_all = X_all.copy()

cat_features = X_all.select_dtypes(['category']).columns.tolist()

for col in cat_features:
    X_all[col] = X_all[col].astype(str)

# -----------------------------
# Stratified K-Fold
# -----------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

accuracy_list = []
macro_f1_list = []
reports = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_all, y_all), 1):

    print(f"\n--- Fold {fold} ---")

    X_tr = X_all.iloc[train_idx]
    X_val_fold = X_all.iloc[val_idx]

    y_tr = y_all.iloc[train_idx]
    y_val_fold = y_all.iloc[val_idx]

    # CatBoost model
    cat_model = CatBoostClassifier(
        iterations=300,
        depth=4,
        learning_rate=0.05,
        loss_function='MultiClass',
        random_seed=42,
        verbose=0
    )

    # Train
    cat_model.fit(
        X_tr,
        y_tr,
        cat_features=cat_features
    )

    # Predict
    y_pred = cat_model.predict(X_val_fold).flatten()

    # Metrics
    acc = accuracy_score(y_val_fold, y_pred)
    macro_f1 = f1_score(y_val_fold, y_pred, average="macro")

    print("Accuracy:", acc)
    print("Macro F1:", macro_f1)
    print(classification_report(y_val_fold, y_pred))

    accuracy_list.append(acc)
    macro_f1_list.append(macro_f1)

    reports.append(classification_report(y_val_fold, y_pred, output_dict=True))

# -----------------------------
# Average Results
# -----------------------------
print("\n===== AVERAGE RESULTS =====")
print("Average Accuracy:", np.mean(accuracy_list))
print("Average Macro F1:", np.mean(macro_f1_list))

# -----------------------------
# Average classification report
# -----------------------------
avg_report = {}

for label in reports[0].keys():

    if label == "accuracy":
        avg_report[label] = np.mean([r[label] for r in reports])
        continue

    avg_report[label] = {}

    for metric in reports[0][label].keys():
        avg_report[label][metric] = np.mean([r[label][metric] for r in reports])

print("\nAverage Classification Report per class:")
print(pd.DataFrame(avg_report).T)


--- Fold 1 ---
Accuracy: 0.6784922394678492
Macro F1: 0.4230740461941907
              precision    recall  f1-score   support

         0.0       0.46      0.24      0.31       411
         1.0       0.71      0.95      0.81      1474
         2.0       0.51      0.08      0.14       370

    accuracy                           0.68      2255
   macro avg       0.56      0.42      0.42      2255
weighted avg       0.63      0.68      0.61      2255


--- Fold 2 ---
Accuracy: 0.6894409937888198
Macro F1: 0.4545458551400116
              precision    recall  f1-score   support

         0.0       0.53      0.25      0.34       412
         1.0       0.71      0.95      0.82      1473
         2.0       0.52      0.13      0.21       369

    accuracy                           0.69      2254
   macro avg       0.59      0.44      0.45      2254
weighted avg       0.65      0.69      0.63      2254


--- Fold 3 ---
Accuracy: 0.6863354037267081
Macro F1: 0.4496031897595949
              pr

In [ ]:
# -----------------------------
# Random forest
# -----------------------------
X_all = pd.concat([X_train, X_val], axis=0)
y_all = pd.concat([y_train, y_val], axis=0)

X_all = X_all.copy()

cat_cols = X_all.select_dtypes(include=['object', 'category']).columns.tolist()

# -----------------------------
# Stratified K-Fold
# -----------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

accuracy_list = []
macro_f1_list = []
reports = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_all, y_all), 1):

    print(f"\n--- Fold {fold} ---")

    X_tr = X_all.iloc[train_idx].copy()
    X_val_fold = X_all.iloc[val_idx].copy()

    y_tr = y_all.iloc[train_idx]
    y_val_fold = y_all.iloc[val_idx]

    # -----------------------------
    # Encode categorical features
    # -----------------------------
    if cat_cols:
        encoder = OrdinalEncoder()

        X_tr[cat_cols] = encoder.fit_transform(X_tr[cat_cols])
        X_val_fold[cat_cols] = encoder.transform(X_val_fold[cat_cols])

    # -----------------------------
    # Random Forest model
    # -----------------------------
    rf_model = RandomForestClassifier(
        n_estimators=300,
        max_depth=6,
        random_state=42,
        class_weight="balanced"
    )

    # Train
    rf_model.fit(X_tr, y_tr)

    # Predict
    y_pred = rf_model.predict(X_val_fold)

    # Metrics
    acc = accuracy_score(y_val_fold, y_pred)
    macro_f1 = f1_score(y_val_fold, y_pred, average="macro")

    print("Accuracy:", acc)
    print("Macro F1:", macro_f1)
    print(classification_report(y_val_fold, y_pred))

    accuracy_list.append(acc)
    macro_f1_list.append(macro_f1)

    reports.append(classification_report(y_val_fold, y_pred, output_dict=True))

# -----------------------------
# AVerage Results
# -----------------------------
print("\n===== AVERAGE RESULTS =====")
print("Average Accuracy:", np.mean(accuracy_list))
print("Average Macro F1:", np.mean(macro_f1_list))

# -----------------------------
# Average classification report
# -----------------------------
avg_report = {}

for label in reports[0].keys():

    if label == "accuracy":
        avg_report[label] = np.mean([r[label] for r in reports])
        continue

    avg_report[label] = {}

    for metric in reports[0][label].keys():
        avg_report[label][metric] = np.mean([r[label][metric] for r in reports])

print("\nAverage Classification Report per class:")
print(pd.DataFrame(avg_report).T)


--- Fold 1 ---
Accuracy: 0.5436807095343681
Macro F1: 0.48888467550371534
              precision    recall  f1-score   support

         0.0       0.32      0.64      0.43       411
         1.0       0.85      0.54      0.66      1474
         2.0       0.33      0.44      0.38       370

    accuracy                           0.54      2255
   macro avg       0.50      0.54      0.49      2255
weighted avg       0.67      0.54      0.57      2255


--- Fold 2 ---
Accuracy: 0.5532386867790594
Macro F1: 0.4980160843489063
              precision    recall  f1-score   support

         0.0       0.34      0.65      0.45       412
         1.0       0.85      0.55      0.67      1473
         2.0       0.33      0.45      0.38       369

    accuracy                           0.55      2254
   macro avg       0.51      0.55      0.50      2254
weighted avg       0.67      0.55      0.58      2254


--- Fold 3 ---
Accuracy: 0.5346051464063887
Macro F1: 0.47629045731817893
              